# Kopp & Lean (2011) — A New, Lower Value of Total Solar Irradiance
## 새로운, 더 낮은 총 태양 복사 조도 값 — 구현 노트북

**Paper / 논문**: Kopp, G. & Lean, J. L. (2011). *Geophysical Research Letters*, 38, L01706. doi:10.1029/2010GL045777

**English.** This notebook reproduces the key climate-relevant calculations of Kopp & Lean (2011): the new TSI absolute value (1360.8 W/m²) versus the legacy value (1365.4 W/m²), the corresponding shift in Earth's disk-averaged absorbed flux and effective temperature, illustrative TSI composite reconstructions with the new offset, and uncertainty propagation into climate sensitivity. We use SI units throughout and `np.trapezoid` for integration.

**한국어.** 이 노트북은 Kopp & Lean(2011)의 핵심 기후 관련 계산을 재현합니다: 새로운 TSI 절대값(1360.8 W/m²) vs. 기존 값(1365.4 W/m²), 지구의 원반 평균 흡수 플럭스와 유효 온도의 해당 이동, 새로운 오프셋이 적용된 TSI 합성 재구성 예시, 기후 민감도로의 불확실성 전파. 전체적으로 SI 단위를 사용하며 적분에는 `np.trapezoid`를 사용합니다.

## Part 1 — Imports and Constants / 임포트 및 상수

**English.** Import scientific Python packages and define the physical constants and TSI values from the paper.

**한국어.** 과학용 파이썬 패키지를 임포트하고 논문의 물리 상수와 TSI 값을 정의합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Physical constants (SI units).
STEFAN_BOLTZMANN = 5.670374419e-8  # W m^-2 K^-4
EARTH_BOND_ALBEDO = 0.30           # dimensionless, planetary albedo

# TSI values from Kopp & Lean (2011).
TSI_OLD = 1365.4   # W/m^2, pre-2011 community standard (cycle-23 minimum reference).
TSI_NEW = 1360.8   # W/m^2, Kopp & Lean (2011) recommended value.
TSI_NEW_SIGMA = 0.5  # W/m^2, 1-sigma uncertainty.
DELTA_TSI = TSI_NEW - TSI_OLD  # -4.6 W/m^2

# Climate-sensitivity baseline (median of IPCC range).
LAMBDA_K_PER_WM2 = 0.8  # K / (W/m^2), equilibrium climate sensitivity.
LAMBDA_SIGMA = 0.3      # K / (W/m^2), 1-sigma.

print(f'TSI_old = {TSI_OLD} W/m^2')
print(f'TSI_new = {TSI_NEW} +/- {TSI_NEW_SIGMA} W/m^2')
print(f'Delta TSI = {DELTA_TSI:+.2f} W/m^2')


## Part 2 — Old vs. New TSI: Disk-Averaged Forcing and Effective Temperature / 이전 vs. 새 TSI: 원반 평균 강제력 및 유효 온도

**English.** Compute the top-of-atmosphere absorbed solar flux $F_{abs} = (1-A) \cdot \text{TSI} / 4$ and the radiative-equilibrium effective temperature $T_{eff} = (F_{abs}/\sigma)^{1/4}$ for both TSI values. The factor $1/4$ is the spherical-Earth disk-to-sphere ratio.

**한국어.** 두 TSI 값에 대한 대기 상단 흡수 태양 플럭스 $F_{abs} = (1-A) \cdot \text{TSI} / 4$ 와 복사 평형 유효 온도 $T_{eff} = (F_{abs}/\sigma)^{1/4}$ 를 계산합니다. 인수 $1/4$ 는 구형 지구의 원반-구 비율입니다.

In [ ]:
def absorbed_flux(tsi: float, albedo: float = EARTH_BOND_ALBEDO) -> float:
    """Compute disk-averaged absorbed solar flux at top of atmosphere.

    Args:
        tsi: Total solar irradiance at 1 AU (W/m^2).
        albedo: Planetary Bond albedo (dimensionless).

    Returns:
        Absorbed flux F_abs = (1 - A) * TSI / 4 in W/m^2.
    """
    return (1.0 - albedo) * tsi / 4.0


def effective_temperature(f_abs: float) -> float:
    """Compute blackbody effective radiating temperature.

    Args:
        f_abs: Absorbed flux at TOA (W/m^2).

    Returns:
        Effective temperature T_eff in Kelvin.
    """
    return (f_abs / STEFAN_BOLTZMANN) ** 0.25


F_abs_old = absorbed_flux(TSI_OLD)
F_abs_new = absorbed_flux(TSI_NEW)
T_eff_old = effective_temperature(F_abs_old)
T_eff_new = effective_temperature(F_abs_new)

print(f'F_abs(old) = {F_abs_old:.3f} W/m^2,  T_eff(old) = {T_eff_old:.3f} K')
print(f'F_abs(new) = {F_abs_new:.3f} W/m^2,  T_eff(new) = {T_eff_new:.3f} K')
print(f'Delta F_abs = {F_abs_new - F_abs_old:+.3f} W/m^2')
print(f'Delta T_eff = {T_eff_new - T_eff_old:+.3f} K')


## Part 3 — Bar Chart: Old vs. New / 막대 그래프: 이전 vs. 새로운

**English.** Visualize the offset between the legacy and Kopp & Lean values for TSI, absorbed flux, and effective temperature.

**한국어.** TSI, 흡수 플럭스, 유효 온도에 대한 기존 값과 Kopp & Lean 값 간 오프셋을 시각화합니다.

In [ ]:
labels = ['TSI (W/m^2)', 'F_abs (W/m^2)', 'T_eff (K)']
old_vals = [TSI_OLD, F_abs_old, T_eff_old]
new_vals = [TSI_NEW, F_abs_new, T_eff_new]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, label, ov, nv in zip(axes, labels, old_vals, new_vals):
    ax.bar(['Old (pre-2011)', 'New (Kopp & Lean 2011)'], [ov, nv],
           color=['steelblue', 'crimson'])
    ax.set_title(label)
    ax.set_ylim(min(ov, nv) * 0.999, max(ov, nv) * 1.001)
    for i, v in enumerate([ov, nv]):
        ax.text(i, v, f'{v:.3f}', ha='center', va='bottom')
fig.suptitle('Old vs. New TSI: Climate-Relevant Quantities / 이전 vs. 새 TSI: 기후 관련 양',
             fontsize=12)
fig.tight_layout()
plt.show()


## Part 4 — Synthetic TSI Composite (PMOD/ACRIM) Reconstructions / 합성 TSI 합성 재구성

**English.** Generate illustrative time series mimicking the PMOD and ACRIM TSI composites. Both follow an 11-year solar cycle with ~1 W/m² peak-to-trough amplitude. We then apply the Kopp & Lean −4.6 W/m² re-baselining and confirm that *relative* variability is preserved while the absolute level shifts.

**한국어.** PMOD 및 ACRIM TSI 합성을 모방한 예시 시계열을 생성합니다. 둘 다 ~1 W/m² peak-to-trough 진폭의 11년 태양주기를 따릅니다. 그런 다음 Kopp & Lean의 −4.6 W/m² 재기준화를 적용하고 절대 수준은 이동하는 동안 *상대* 변동성은 보존됨을 확인합니다.

In [ ]:
# Synthesize a TSI composite from 1979 to 2010.
rng = np.random.default_rng(42)
years = np.linspace(1979.0, 2010.0, 32 * 12)  # monthly cadence
cycle_period = 11.0
cycle_amplitude = 0.5  # W/m^2 (half peak-to-trough = 0.5 ~ amplitude 1.0 p2p)

# Sinusoid centered at the cycle minimum 1986 plus small drift differences.
phase = 2.0 * np.pi * (years - 1986.0) / cycle_period
common_signal = cycle_amplitude * np.sin(phase)

# PMOD: stays flat across 1986/1996 minima.
pmod_drift = 0.0 * (years - 1990.0)
pmod_noise = rng.normal(0, 0.05, size=years.size)
tsi_pmod_legacy = TSI_OLD + common_signal + pmod_drift + pmod_noise

# ACRIM: ~0.04% upward trend across the same period.
acrim_drift = 0.04e-2 * TSI_OLD * (years - 1986.0) / 24.0
acrim_noise = rng.normal(0, 0.07, size=years.size)
tsi_acrim_legacy = TSI_OLD + common_signal + acrim_drift + acrim_noise

# Apply Kopp & Lean re-baselining.
tsi_pmod_new = tsi_pmod_legacy + DELTA_TSI
tsi_acrim_new = tsi_acrim_legacy + DELTA_TSI

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(years, tsi_pmod_legacy, label='PMOD (legacy)', color='steelblue', lw=1.0)
axes[0].plot(years, tsi_acrim_legacy, label='ACRIM (legacy)', color='darkorange', lw=1.0)
axes[0].axhline(TSI_OLD, color='gray', ls='--', alpha=0.6, label=f'Legacy = {TSI_OLD}')
axes[0].set_ylabel('TSI (W/m^2)')
axes[0].set_title('Legacy absolute level / 기존 절대 수준')
axes[0].legend(loc='lower right')

axes[1].plot(years, tsi_pmod_new, label='PMOD (rebaselined)', color='steelblue', lw=1.0)
axes[1].plot(years, tsi_acrim_new, label='ACRIM (rebaselined)', color='darkorange', lw=1.0)
axes[1].axhline(TSI_NEW, color='gray', ls='--', alpha=0.6, label=f'New = {TSI_NEW}')
axes[1].set_ylabel('TSI (W/m^2)')
axes[1].set_xlabel('Year')
axes[1].set_title('Kopp & Lean (2011) absolute level / Kopp & Lean(2011) 절대 수준')
axes[1].legend(loc='lower right')
fig.tight_layout()
plt.show()

print(f'PMOD legacy mean: {tsi_pmod_legacy.mean():.2f} W/m^2')
print(f'PMOD rebaselined mean: {tsi_pmod_new.mean():.2f} W/m^2')
print(f'PMOD legacy std (variability): {tsi_pmod_legacy.std():.4f} W/m^2')
print(f'PMOD rebaselined std (variability): {tsi_pmod_new.std():.4f} W/m^2  (unchanged)')


## Part 5 — Time-Integrated Energy via np.trapezoid / 시간 적분 에너지

**English.** Use `np.trapezoid` to integrate the disk-averaged absorbed flux over the simulated 1979–2010 interval, comparing legacy and rebaselined cases. The time-integrated energy difference quantifies the cumulative offset that climate-energy budgets would carry under the legacy versus new TSI scale.

**한국어.** `np.trapezoid`를 사용하여 시뮬레이션된 1979–2010 구간 동안의 원반 평균 흡수 플럭스를 적분하고, 기존 및 재기준화 사례를 비교합니다. 시간 적분된 에너지 차이는 기후 에너지 수지가 기존 vs. 새로운 TSI 스케일 하에서 운반할 누적 오프셋을 정량화합니다.

In [ ]:
# Convert years to seconds for energy units (J/m^2).
seconds_per_year = 365.25 * 24.0 * 3600.0
time_seconds = (years - years[0]) * seconds_per_year

# Use the PMOD composite as the reference TSI series.
f_abs_legacy_series = (1.0 - EARTH_BOND_ALBEDO) * tsi_pmod_legacy / 4.0
f_abs_new_series = (1.0 - EARTH_BOND_ALBEDO) * tsi_pmod_new / 4.0

energy_legacy = np.trapezoid(f_abs_legacy_series, time_seconds)
energy_new = np.trapezoid(f_abs_new_series, time_seconds)
delta_energy = energy_new - energy_legacy

print(f'Time-integrated F_abs (legacy)   = {energy_legacy:.3e} J/m^2')
print(f'Time-integrated F_abs (new)      = {energy_new:.3e} J/m^2')
print(f'Difference                       = {delta_energy:+.3e} J/m^2')
duration_years = (years[-1] - years[0])
print(f'Span: {duration_years:.1f} years')
print(f'Average power offset: {delta_energy / (duration_years * seconds_per_year):+.3f} W/m^2')
print('Confirms expected -0.805 W/m^2 disk-averaged offset.')


## Part 6 — Uncertainty Propagation to Climate Sensitivity / 기후 민감도로의 불확실성 전파

**English.** Combine the TSI absolute uncertainty (0.5 W/m²), albedo uncertainty (0.01), and climate-sensitivity uncertainty (0.3 K/(W/m²)) to compute the 1-σ uncertainty in equilibrium temperature change driven by the absolute TSI offset. Show that climate-sensitivity uncertainty dominates.

**한국어.** TSI 절대 불확실성(0.5 W/m²), 알베도 불확실성(0.01), 기후 민감도 불확실성(0.3 K/(W/m²))을 결합하여 절대 TSI 오프셋이 구동하는 평형 온도 변화의 1-σ 불확실성을 계산합니다. 기후 민감도 불확실성이 지배적임을 보입니다.

In [ ]:
def equilibrium_dT_and_sigma(delta_tsi: float,
                              tsi_sigma: float,
                              albedo: float,
                              albedo_sigma: float,
                              lam: float,
                              lam_sigma: float):
    """Propagate uncertainties from TSI/albedo/climate-sensitivity to delta T_eq.

    Computes delta_T = lam * (1 - A) * delta_tsi / 4 and the 1-sigma uncertainty
    via standard partial-derivative error propagation.

    Args:
        delta_tsi: Change in TSI (W/m^2).
        tsi_sigma: 1-sigma uncertainty on TSI (W/m^2).
        albedo: Planetary Bond albedo (dimensionless).
        albedo_sigma: 1-sigma uncertainty on albedo (dimensionless).
        lam: Climate sensitivity (K per W/m^2).
        lam_sigma: 1-sigma uncertainty on climate sensitivity.

    Returns:
        Tuple (delta_T, sigma_delta_T, contribution_dict) in Kelvin.
    """
    delta_F = (1.0 - albedo) * delta_tsi / 4.0
    delta_T = lam * delta_F

    # Partial derivatives.
    dT_dTSI = lam * (1.0 - albedo) / 4.0
    dT_dA = -lam * delta_tsi / 4.0
    dT_dlam = delta_F

    # Variance contributions.
    var_tsi = (dT_dTSI * tsi_sigma) ** 2
    var_a = (dT_dA * albedo_sigma) ** 2
    var_lam = (dT_dlam * lam_sigma) ** 2
    sigma_delta_T = np.sqrt(var_tsi + var_a + var_lam)

    contributions = {
        'TSI': np.sqrt(var_tsi),
        'Albedo': np.sqrt(var_a),
        'Climate sensitivity': np.sqrt(var_lam),
    }
    return delta_T, sigma_delta_T, contributions


delta_T, sigma_dT, parts = equilibrium_dT_and_sigma(
    delta_tsi=DELTA_TSI,
    tsi_sigma=TSI_NEW_SIGMA,
    albedo=EARTH_BOND_ALBEDO,
    albedo_sigma=0.01,
    lam=LAMBDA_K_PER_WM2,
    lam_sigma=LAMBDA_SIGMA,
)
print(f'Delta T_eq from K&L offset = {delta_T:+.3f} +/- {sigma_dT:.3f} K (1-sigma)')
print('Contributions to sigma(Delta T):')
for k, v in parts.items():
    print(f'  {k:25s}: {v:.4f} K')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(parts.keys(), parts.values(), color=['steelblue', 'darkorange', 'crimson'])
ax.set_ylabel('1-sigma contribution to Delta T (K)')
ax.set_title('Uncertainty budget for K&L absolute offset / K&L 절대 오프셋 불확실성 수지')
fig.tight_layout()
plt.show()


## Part 7 — Conclusions / 결론

**English.**
1. Kopp & Lean's recommended TSI 1360.8 ± 0.5 W/m² lowers the legacy value by 4.6 W/m² (~0.34%).
2. The disk-averaged absorbed flux drops by ~0.81 W/m², shifting the radiative-equilibrium effective temperature by ~−0.22 K.
3. Because the offset is constant in time, *cycle-to-cycle* TSI variability and any TSI-anomaly-based climate analysis are unaffected.
4. Climate-sensitivity uncertainty (~0.3 K/(W/m²)) dominates the propagated equilibrium-temperature uncertainty — the TSI absolute level is no longer the bottleneck.
5. The new value has been adopted by IPCC AR5/AR6 and CMIP6 as the standard solar boundary condition.

**한국어.**
1. Kopp & Lean의 권장 TSI 1360.8 ± 0.5 W/m²는 기존 값을 4.6 W/m²(~0.34%) 낮춥니다.
2. 원반 평균 흡수 플럭스는 ~0.81 W/m² 감소하여, 복사 평형 유효 온도를 ~−0.22 K 이동시킵니다.
3. 오프셋이 시간에 대해 일정하므로 *주기 간* TSI 변동성과 TSI 변칙 기반 기후 분석은 영향받지 않습니다.
4. 기후 민감도 불확실성(~0.3 K/(W/m²))이 전파된 평형 온도 불확실성을 지배합니다 — TSI 절대 수준은 더 이상 병목이 아닙니다.
5. 새로운 값은 IPCC AR5/AR6 및 CMIP6에서 표준 태양 경계 조건으로 채택되었습니다.